# 06 — Fase 4: Validazione Downstream sulla Rete Siamese (`SiameseV3`) con FCGR Ottimizzate

## Scopo del Notebook
Verificare sperimentalmente la nostra ipotesi di tesi: **mantenendo invariata l'architettura `SiameseV3`, gli iperparametri e gli split (`Train`, `Val Pool`, `Test Pool`) del Notebook `04`**, valutiamo come cambia l'apprendimento della rete sostituendo la FCGR Baseline ($k=6$) con le due varianti ottimizzate:
1. **EsperimentoOttimizzato 1 (`siamese_opt_rc_log_k6`)**:
   - Input `(1, 64, 64)` generato con **Bitwise Vectorized + Reverse-Complement (Strand-Invariant) + Normalizzazione `log1p`** e salvato in `uint8` (**`228.9 MB` invece di `915.5 MB`, -75% memoria**).
   - Isola l'effetto dell'**invarianza al filamento** e della **correzione del bias di lunghezza** a parità esatta di 1 canale (`807.264` parametri).
2. **Esperimento Ottimizzato 2 (`siamese_opt_multiscale_k456`)**:
   - Input `(3, 64, 64)` generato con **Stack Frattale Multi-Scala $k=(4, 5, 6)$ + Reverse-Complement + `log1p`** e salvato in `uint8` (**`686.6 MB` invece di `915.5 MB`, -25% memoria** e sparsità spaziale ridotta da ~90% a <20%).
   - Fornisce a `SiameseV3(in_channels=3)` contemporaneamente il contesto denso ($k=4, 5$) e la precisione spaziale ($k=6$).

In [ ]:
# Su Google Colab esegui il mount del Drive:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('MANIFEST_PATH:', MANIFEST_PATH, '-> exists:', MANIFEST_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())
import copy
import math
import torch
from torch.utils.data import DataLoader
from numpy.lib.format import open_memmap
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from src.fcgr import (
    load_fasta_sequences,
    compute_fcgr_vectorized,
    normalize_fcgr_log,
    quantize_fcgr_uint8,
    compute_multiscale_fcgr,
    plot_siamese_pair_feedback,
)
from src.dataset import (
    RandomSiamesePairDataset,
    BalancedSiamesePairDataset,
    BalancedSemiHardCandidateDataset,
    SingleFCGRDataset,
)
from src.models import (
    SiameseV3,
    EuclideanContrastiveLoss,
    select_semihard_negatives,
    evaluate_pairwise,
    extract_embeddings,
    compute_topk_similarity_matrix,
)

## 1. Configurazione Controllata (Identica al Notebook `04` Baseline)
Usiamo esattamente gli stessi seed, gli stessi pool (`50.000` Train, `5.000` Val, `5.000` Test) e gli stessi iperparametri di `04_siamese_model.ipynb`.

In [ ]:
K = 6
RANDOM_STATE = 42
N_CLASSES = 2
EMBEDDING_DIM = 128
EUCLIDEAN_MARGIN = 1.25

WARMUP_EPOCHS = 3
MAX_SEMIHARD_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

BALANCED_BATCH_SIZE = 64
VAL_BATCH_SIZE = 128
VAL_PAIRS = 10_000
TEST_PAIRS = 10_000
N_NEGATIVE_CANDIDATES = 6

WARMUP_LR = 5e-4
SEMIHARD_LR = 1e-4
WEIGHT_DECAY = 1e-4

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
print('Device      :', DEVICE, f'({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else '')
print('AMP Enabled :', AMP_ENABLED)

VAL_POOL_PATH = PROCESSED_DIR / 'siamese_val_pair_pool.tsv'
TEST_POOL_PATH = PROCESSED_DIR / 'siamese_test_pair_pool.tsv'
FCGR_CACHE_DIR = PROCESSED_DIR / 'fcgr_cache'
FCGR_INDEX_PATH = FCGR_CACHE_DIR / 'fcgr_k6_index.tsv'

metadata = pd.read_csv(MANIFEST_PATH, sep='\t', dtype={'id': str})
metadata['class_id'] = metadata['class_id'].astype(int)
train_metadata = metadata[metadata['split_cluster'] == 'train'].copy().reset_index(drop=True)
val_metadata = pd.read_csv(VAL_POOL_PATH, sep='\t', dtype={'id': str})
val_metadata['class_id'] = val_metadata['class_id'].astype(int)
test_metadata = pd.read_csv(TEST_POOL_PATH, sep='\t', dtype={'id': str})
test_metadata['class_id'] = test_metadata['class_id'].astype(int)

fcgr_index = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
id_to_fcgr_row = dict(zip(fcgr_index['id'], fcgr_index['fcgr_row']))
ordered_ids = fcgr_index.sort_values('fcgr_row')['id'].tolist()
N_TOTAL_SAMPLES = len(ordered_ids)
print(f'Sequenze indicizzate totali: {N_TOTAL_SAMPLES:,} (Train={len(train_metadata):,}, Val={len(val_metadata):,}, Test={len(test_metadata):,})')

## 2. Precomputazione Veloce delle Cache FCGR Ottimizzate (`uint8`)
Grazie agli algoritmi vettorizzati bitwise in `src/fcgr.py`, generiamo in pochi secondi le cache per le `60.000` sequenze:
1. `fcgr_opt_rc_log_k6_u8.npy` — shape `(60000, 64, 64)`, `uint8` (`228.9 MB` invece di `915.5 MB`).
2. `fcgr_opt_multiscale_k456_u8.npy` — shape `(60000, 3, 64, 64)`, `uint8` (`686.6 MB` invece di `915.5 MB`).

In [ ]:
RC_LOG_CACHE_PATH = FCGR_CACHE_DIR / 'fcgr_opt_rc_log_k6_u8.npy'
RC_LOG_META_PATH = FCGR_CACHE_DIR / 'fcgr_opt_rc_log_k6_u8_meta.json'
MS_CACHE_PATH = FCGR_CACHE_DIR / 'fcgr_opt_multiscale_k456_u8.npy'
MS_META_PATH = FCGR_CACHE_DIR / 'fcgr_opt_multiscale_k456_u8_meta.json'

need_rc = not RC_LOG_CACHE_PATH.exists()
need_ms = not MS_CACHE_PATH.exists()

if need_rc or need_ms:
    print('Caricamento delle 60.000 sequenze dal file FASTA...')
    t_fa = time.perf_counter()
    seq_store = load_fasta_sequences(FASTA_PATH, ordered_ids)
    print(f'Caricate {len(seq_store):,} sequenze in {time.perf_counter() - t_fa:.1f}s')

if need_rc:
    print('\nGenerazione Cache 1: Bitwise RC + Log1p (k=6, uint8, 228.9 MB)...')
    tmp_rc = FCGR_CACHE_DIR / 'tmp_rc_log_u8.npy'
    mm_rc = open_memmap(tmp_rc, mode='w+', dtype=np.uint8, shape=(N_TOTAL_SAMPLES, 64, 64))
    t0 = time.perf_counter()
    for r_idx, sid in enumerate(ordered_ids):
        c_rc, _, _ = compute_fcgr_vectorized(seq_store[sid], 6, return_distinct=True, include_reverse_complement=True)
        mm_rc[r_idx] = quantize_fcgr_uint8(normalize_fcgr_log(c_rc))
        if (r_idx + 1) % 10_000 == 0 or (r_idx + 1) == N_TOTAL_SAMPLES:
            el = time.perf_counter() - t0
            print(f'  [{r_idx+1:6,d}/{N_TOTAL_SAMPLES:,}] ({(r_idx+1)/max(el,1e-6):.0f} seq/s, {el:.1f}s)')
    tot_rc_s = time.perf_counter() - t0
    mm_rc.flush()
    del mm_rc
    tmp_rc.replace(RC_LOG_CACHE_PATH)
    with open(RC_LOG_META_PATH, 'w', encoding='utf-8') as f:
        json.dump({'variant': 'opt_rc_log_k6_u8', 'shape': [N_TOTAL_SAMPLES, 64, 64], 'dtype': 'uint8', 'size_mb': RC_LOG_CACHE_PATH.stat().st_size / (1024**2), 'gen_seconds': tot_rc_s, 'ms_per_seq': tot_rc_s * 1000.0 / N_TOTAL_SAMPLES}, f, indent=4)
    print(f'Cache 1 completata in {tot_rc_s:.1f}s ({tot_rc_s*1000/N_TOTAL_SAMPLES:.3f} ms/seq)!')
else:
    print('Cache 1 (RC+Log k=6 uint8) già presente su disco.')

if need_ms:
    print('\nGenerazione Cache 2: Multi-Scala Frattale k=(4,5,6) RC + Log1p (uint8, 686.6 MB)...')
    tmp_ms = FCGR_CACHE_DIR / 'tmp_ms_k456_u8.npy'
    mm_ms = open_memmap(tmp_ms, mode='w+', dtype=np.uint8, shape=(N_TOTAL_SAMPLES, 3, 64, 64))
    t0 = time.perf_counter()
    for r_idx, sid in enumerate(ordered_ids):
        ms_f32, _ = compute_multiscale_fcgr(seq_store[sid], ks=(4, 5, 6), target_k=6, include_reverse_complement=True, log_normalize=True)
        mm_ms[r_idx] = quantize_fcgr_uint8(ms_f32)
        if (r_idx + 1) % 10_000 == 0 or (r_idx + 1) == N_TOTAL_SAMPLES:
            el = time.perf_counter() - t0
            print(f'  [{r_idx+1:6,d}/{N_TOTAL_SAMPLES:,}] ({(r_idx+1)/max(el,1e-6):.0f} seq/s, {el:.1f}s)')
    tot_ms_s = time.perf_counter() - t0
    mm_ms.flush()
    del mm_ms
    tmp_ms.replace(MS_CACHE_PATH)
    with open(MS_META_PATH, 'w', encoding='utf-8') as f:
        json.dump({'variant': 'opt_multiscale_k456_u8', 'shape': [N_TOTAL_SAMPLES, 3, 64, 64], 'dtype': 'uint8', 'size_mb': MS_CACHE_PATH.stat().st_size / (1024**2), 'gen_seconds': tot_ms_s, 'ms_per_seq': tot_ms_s * 1000.0 / N_TOTAL_SAMPLES}, f, indent=4)
    print(f'Cache 2 completata in {tot_ms_s:.1f}s ({tot_ms_s*1000/N_TOTAL_SAMPLES:.3f} ms/seq)!')
else:
    print('Cache 2 (Multi-Scala k=4,5,6 uint8) già presente su disco.')

## 3. Pipeline Unificata di Training e Validazione Downstream per `SiameseV3`
Definiamo la funzione `run_siamese_experiment(...)` che esegue esattamente lo stesso protocollo del Notebook `04` (3 epoche di Random Warm-up + fino a 15 epoche di Dynamic Semi-Hard Negative Mining + valutazione Pairwise e Classificazione Binaria con Reference Bank `top_k=20`).

In [ ]:
WARMUP_BATCHES_PER_EPOCH = math.ceil(len(train_metadata) / BALANCED_BATCH_SIZE)
WARMUP_PAIRS_PER_EPOCH = WARMUP_BATCHES_PER_EPOCH * BALANCED_BATCH_SIZE
MINING_BATCHES_PER_EPOCH = math.ceil((WARMUP_PAIRS_PER_EPOCH / 2) / BALANCED_BATCH_SIZE)

def train_random_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    model.train()
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances = [], []
    start_time = time.perf_counter()
    for batch in loader:
        x1 = batch['x1'].to(DEVICE, non_blocking=True)
        x2 = batch['x2'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            z1, z2 = model(x1, x2)
        loss, distances = criterion(z1, z2, target)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        bs = target.shape[0]
        total_loss += loss.item() * bs
        total_samples += bs
        all_targets.append(target.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
    targets = np.concatenate(all_targets)
    distances = np.concatenate(all_distances)
    return {'loss': total_loss / total_samples, 'auc': float(roc_auc_score(targets, -distances)), 'seconds': time.perf_counter() - start_time}

def train_dynamic_semihard_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances, semihard_fractions = [], [], []
    start_time = time.perf_counter()
    for batch in loader:
        anchor = batch['anchor'].to(DEVICE, non_blocking=True)
        positive = batch['positive'].to(DEVICE, non_blocking=True)
        negatives = batch['negatives'].to(DEVICE, non_blocking=True)
        selected_negative, mining_stats = select_semihard_negatives(
            model=model, anchor=anchor, positive=positive, negatives=negatives,
            margin=EUCLIDEAN_MARGIN, amp_enabled=AMP_ENABLED,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        bs = anchor.shape[0]
        train_images = torch.cat([anchor, positive, selected_negative], dim=0)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            embeddings = model.encoder(train_images)
        z_a, z_p, z_n = embeddings[:bs], embeddings[bs:2*bs], embeddings[2*bs:]
        z1 = torch.cat([z_a, z_a], dim=0)
        z2 = torch.cat([z_p, z_n], dim=0)
        targets = torch.cat([torch.ones(bs, device=DEVICE), torch.zeros(bs, device=DEVICE)], dim=0)
        loss, distances = criterion(z1, z2, targets)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        n_pairs = targets.shape[0]
        total_loss += loss.detach().item() * n_pairs
        total_samples += n_pairs
        all_targets.append(targets.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
        semihard_fractions.append(mining_stats['semihard_fraction'])
    targets_np = np.concatenate(all_targets)
    distances_np = np.concatenate(all_distances)
    return {'loss': total_loss / total_samples, 'auc': float(roc_auc_score(targets_np, -distances_np)), 'semihard_fraction': float(np.mean(semihard_fractions)), 'seconds': time.perf_counter() - start_time}

def run_siamese_experiment(exp_name: str, fcgr_cache_path: Path, in_channels: int):
    print('=' * 95)
    print(f'AVVIO ESPERIMENTO: {exp_name} (in_channels={in_channels})')
    print('=' * 95)
    out_dir = ARTIFACTS_DIR / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    fcgr_mm = np.load(fcgr_cache_path, mmap_mode='r')
    cache_mb = fcgr_cache_path.stat().st_size / (1024 ** 2)
    print(f'Cache aperta: {fcgr_cache_path.name} | shape={fcgr_mm.shape} | dtype={fcgr_mm.dtype} | {cache_mb:.1f} MB')

    set_seed(RANDOM_STATE)
    warmup_ds = BalancedSiamesePairDataset(train_metadata, fcgr_mm, id_to_fcgr_row, WARMUP_BATCHES_PER_EPOCH, BALANCED_BATCH_SIZE, 0.50, RANDOM_STATE)
    mining_ds = BalancedSemiHardCandidateDataset(train_metadata, fcgr_mm, id_to_fcgr_row, MINING_BATCHES_PER_EPOCH, BALANCED_BATCH_SIZE, N_NEGATIVE_CANDIDATES, RANDOM_STATE)
    val_ds = RandomSiamesePairDataset(val_metadata, fcgr_mm, id_to_fcgr_row, VAL_PAIRS, 0.50, RANDOM_STATE + 50_000)
    test_ds = RandomSiamesePairDataset(test_metadata, fcgr_mm, id_to_fcgr_row, TEST_PAIRS, 0.50, RANDOM_STATE + 60_000)

    warmup_loader = DataLoader(warmup_ds, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    mining_loader = DataLoader(mining_ds, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    val_loader = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
    test_loader = DataLoader(test_ds, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)

    model = SiameseV3(embedding_dim=EMBEDDING_DIM, in_channels=in_channels).to(DEVICE)
    criterion = EuclideanContrastiveLoss(margin=EUCLIDEAN_MARGIN)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Parametri trainabili SiameseV3: {n_params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=WARMUP_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

    print('\n--- FASE 1: RANDOM-PAIR WARM-UP ---')
    for epoch in range(1, WARMUP_EPOCHS + 1):
        tr = train_random_epoch(model, warmup_loader, warmup_ds, criterion, optimizer, scaler, epoch)
        va = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
        print(f"Warm-up {epoch:02d}/{WARMUP_EPOCHS} | train AUC {tr['auc']:.4f} | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s")

    print('\n--- FASE 2: DYNAMIC SEMI-HARD NEGATIVE MINING ---')
    ckpt_path = out_dir / 'best_model.pt'
    optimizer = torch.optim.AdamW(model.parameters(), lr=SEMIHARD_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_metrics = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
    best_auc = best_metrics['auc']
    best_epoch = 0
    epochs_no_improve = 0
    history = []
    torch.save({'epoch': 0, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics}, ckpt_path)

    for epoch in range(1, MAX_SEMIHARD_EPOCHS + 1):
        tr = train_dynamic_semihard_epoch(model, mining_loader, mining_ds, criterion, optimizer, scaler, WARMUP_EPOCHS + epoch)
        va = evaluate_pairwise(model, val_loader, criterion, DEVICE, AMP_ENABLED)
        history.append({'epoch': epoch, 'train_auc': tr['auc'], 'semihard_fraction': tr['semihard_fraction'], **va})
        if va['auc'] > best_auc + MIN_DELTA:
            best_auc = va['auc']
            best_epoch = epoch
            best_metrics = copy.deepcopy(va)
            epochs_no_improve = 0
            torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics}, ckpt_path)
            marker = ' <-- BEST'
        else:
            epochs_no_improve += 1
            marker = ''
        print(f"Epoch {epoch:02d}/{MAX_SEMIHARD_EPOCHS} | train AUC {tr['auc']:.4f} | semi-hard {tr['semihard_fraction']*100:.1f}% | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s{marker}")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print('Early stopping attivato.')
            break

    pd.DataFrame(history).to_csv(out_dir / 'training_history.tsv', sep='\t', index=False)
    best_ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt['model_state_dict'])
    model.eval()

    @torch.no_grad()
    def collect_dists(loader):
        t_all, d_all = [], []
        for b in loader:
            x1 = b['x1'].to(DEVICE, non_blocking=True)
            x2 = b['x2'].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
                z1, z2 = model(x1, x2)
            d = torch.linalg.vector_norm(z1.float() - z2.float(), ord=2, dim=1)
            t_all.append(b['target'].numpy())
            d_all.append(d.cpu().numpy())
        return np.concatenate(t_all), np.concatenate(d_all)

    val_t, val_d = collect_dists(val_loader)
    test_t, test_d = collect_dists(test_loader)
    thrs = np.linspace(val_d.min(), val_d.max(), 400)
    best_thr = float(thrs[int(np.argmax([f1_score(val_t, (val_d <= t).astype(int)) for t in thrs]))])
    test_preds_p = (test_d <= best_thr).astype(int)
    test_scores_p = -test_d
    pos_d, neg_d = test_d[test_t == 1], test_d[test_t == 0]
    gap_d = float(neg_d.mean() - pos_d.mean())
    d_prime_test = float(gap_d / np.sqrt(0.5 * (pos_d.var() + neg_d.var()) + 1e-12))

    pairwise_metrics = {
        'pairwise_roc_auc': float(roc_auc_score(test_t, test_scores_p)),
        'pairwise_pr_auc': float(average_precision_score(test_t, test_scores_p)),
        'pairwise_f1': float(f1_score(test_t, test_preds_p)),
        'pairwise_macro_f1': float(f1_score(test_t, test_preds_p, average='macro')),
        'pairwise_precision': float(precision_score(test_t, test_preds_p)),
        'pairwise_recall': float(recall_score(test_t, test_preds_p)),
        'pairwise_accuracy': float(accuracy_score(test_t, test_preds_p)),
        'd_pos_mean': float(pos_d.mean()),
        'd_neg_mean': float(neg_d.mean()),
        'distance_gap': gap_d,
        'd_prime': d_prime_test,
        'threshold': best_thr,
    }

    train_emb_loader = DataLoader(SingleFCGRDataset(train_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    test_emb_loader = DataLoader(SingleFCGRDataset(test_metadata, fcgr_mm, id_to_fcgr_row), batch_size=256, shuffle=False)
    train_emb, train_lbl = extract_embeddings(model, train_emb_loader, DEVICE, AMP_ENABLED)
    test_emb, test_lbl = extract_embeddings(model, test_emb_loader, DEVICE, AMP_ENABLED)

    ref_per_class = min(5000, int((train_lbl == 0).sum()), int((train_lbl == 1).sum()))
    rng = np.random.default_rng(RANDOM_STATE)
    ref_idx = np.concatenate([
        rng.choice(np.where(train_lbl == 0)[0], size=ref_per_class, replace=False),
        rng.choice(np.where(train_lbl == 1)[0], size=ref_per_class, replace=False),
    ])
    sim_mat = compute_topk_similarity_matrix(test_emb, train_emb[ref_idx], train_lbl[ref_idx], n_classes=2, top_k=20)
    hotspot_score = sim_mat[:, 1] - sim_mat[:, 0]
    test_pred_cls = np.argmax(sim_mat, axis=1)

    cls_metrics = {
        'cls_roc_auc': float(roc_auc_score(test_lbl, hotspot_score)),
        'cls_pr_auc': float(average_precision_score(test_lbl, hotspot_score)),
        'cls_f1': float(f1_score(test_lbl, test_pred_cls)),
        'cls_macro_f1': float(f1_score(test_lbl, test_pred_cls, average='macro')),
        'cls_precision': float(precision_score(test_lbl, test_pred_cls)),
        'cls_recall': float(recall_score(test_lbl, test_pred_cls)),
        'cls_balanced_accuracy': float(balanced_accuracy_score(test_lbl, test_pred_cls)),
    }

    summary = {
        **pairwise_metrics,
        **cls_metrics,
        'k': K,
        'in_channels': in_channels,
        'cache_mb': round(cache_mb, 2),
        'train_samples': len(train_metadata),
        'best_epoch': best_epoch,
    }
    pd.DataFrame([summary]).to_csv(out_dir / f'{exp_name}_final_metrics.csv', index=False)
    with open(out_dir / f'{exp_name}_final_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=4)

    print(f'\n=== RISULTATI TEST SET: {exp_name} ===')
    display(pd.DataFrame([summary]))
    print(classification_report(test_lbl, test_pred_cls, target_names=['Stable (0)', 'Hotspot (1)'], digits=4))
    return model, test_loader, summary, (test_t, test_d, pos_d, neg_d, test_lbl, hotspot_score)

## 4. Esecuzione Esperimento Ottimizzato 1: `siamese_opt_rc_log_k6` (1 Canale: Bitwise + RC + Log1p, `uint8`)

In [ ]:
model_rc, test_loader_rc, summary_rc, curves_rc = run_siamese_experiment(
    exp_name='siamese_opt_rc_log_k6',
    fcgr_cache_path=RC_LOG_CACHE_PATH,
    in_channels=1,
)

## 5. Esecuzione Esperimento Ottimizzato 2: `siamese_opt_multiscale_k456` (3 Canali: Multi-Scala $k=4,5,6$ + RC + Log1p, `uint8`)

In [ ]:
model_ms, test_loader_ms, summary_ms, curves_ms = run_siamese_experiment(
    exp_name='siamese_opt_multiscale_k456',
    fcgr_cache_path=MS_CACHE_PATH,
    in_channels=3,
)

## 6. Feedback Visivo Coppie Multi-Scala Post-Training (con Verifica Soglia)
Ispezioniamo sul Test Set una coppia **Positiva ben classificata** ($d \le \text{threshold}$) e una **Negativa ben separata** ($d > \text{threshold}$) con la rappresentazione Multi-Scala.

In [ ]:
test_batch_ms = next(iter(test_loader_ms))
thr_ms = summary_ms['threshold']
with torch.no_grad():
    z1_b, z2_b = model_ms(test_batch_ms['x1'].to(DEVICE), test_batch_ms['x2'].to(DEVICE))
    dists_b = torch.linalg.vector_norm(z1_b.float() - z2_b.float(), ord=2, dim=1).cpu().numpy()
targets_b = test_batch_ms['target'].numpy()

# Selezioniamo 1 coppia Positiva rappresentativa (vicina a d_pos_mean) e 1 Negativa rappresentativa (vicina a d_neg_mean)
pos_indices = np.where((targets_b == 1) & (dists_b <= thr_ms))[0]
neg_indices = np.where((targets_b == 0) & (dists_b > thr_ms))[0]
pos_idx = int(pos_indices[0]) if len(pos_indices) > 0 else int(np.where(targets_b == 1)[0][0])
neg_idx = int(neg_indices[0]) if len(neg_indices) > 0 else int(np.where(targets_b == 0)[0][0])

for idx, desc in [(pos_idx, 'Multi-Scala Positiva'), (neg_idx, 'Multi-Scala Negativa')]:
    plot_siamese_pair_feedback(
        x1=test_batch_ms['x1'][idx],
        x2=test_batch_ms['x2'][idx],
        target=float(targets_b[idx]),
        label1=f'{desc} (Ramo 1)',
        label2=f'{desc} (Ramo 2)',
        embedding_distance=float(dists_b[idx]),
        threshold=thr_ms,
    )